# Script triagem do texto completo

## Configurações

In [ ]:
import csv

with open('/content/drive/MyDrive/Colab Notebooks/articles_title_abs.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

['list', 'Title:', 'title', 'Abstract:\xa0', 'abstract', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['1', 'Title:', 'ServLessSense: Serverless Smell Detection Tool', 'Abstract:\xa0', 'Serverless computing has gained widespread adoption due to its scalability, cost-efficiency, and abstraction of infrastructure management. However, the shift toward event-driven, function-based architectures introduces new code quality challenges and development practices that differ from traditional paradigms. While recent research has identified serverless-specific bad practices commonly referred to as “smells,” there remains a lack of automated tools to support their detection and remediation. This paper presents ServLessSense, a tool designed to detect code smells automatically in serverless applications written in JavaScript and TypeScript. Built using a custom ESLint plugin, ServLessSense identifies five serverless-specific smells, provides visualizations through an interactive dash

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_SECRET"

assert os.getenv("OPENAI_API_KEY")

In [ ]:
import csv, time, os
from datetime import datetime, timezone
from openai import OpenAI, APITimeoutError, APIConnectionError
import httpx

MODEL = "gpt-4o-2024-08-06"
TEMPERATURE = 0
MAX_TOKENS = 3

TOPIC = "JavaScript/TypeScript code smell detection tools and techniques."
CRITERIA = [
    "No detection tool/technique/metric is proposed, evaluated, or analyzed.",
    "Outside maintenance/evolution/quality contexts relevant to code smell detection.",
    "Not about JavaScript/TypeScript or their ecosystems.",
    "Does not address code smells or anti-patterns.",
    "Full text unavailable or publication after 2025 (per protocol).",
]

def make_prompt(title, abstract):
    crit = "\n".join(f"{i+1}) {c}" for i,c in enumerate(CRITERIA))
    return (
        f"You are screening papers for a systematic review on '{TOPIC}'.\n"
        "Decide if the article should be included or excluded from the systematic review.\n"
        "Only answer INCLUDE or EXCLUDE. Be lenient; prefer including by mistake rather than excluding by mistake.\n"
        "Exclude the article if any of the following are true:\n"
        f"{crit}\n\n"
        f"Title: \"{title.strip()}\"\n"
        f"Abstract: \"{abstract.strip()}\""
    )

def classify_row(client, title, abstract, retries=3):
    prompt = make_prompt(title, abstract)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[{"role": "user", "content": prompt}],
            )
            text = resp.choices[0].message.content.strip().upper()
            label = "INCLUDE" if "INCLUDE" in text and "EXCLUDE" not in text else "EXCLUDE"
            return label, text
        except (APITimeoutError, APIConnectionError) as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"Timeout/connection error on attempt {attempt+1}/{retries}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"Failed after {retries} attempts: {e}")
                return "EXCLUDE", f"EXCLUDE (API timeout)"

def run(input_csv, output_csv, rate_limit_per_sec=1):
    timeout_config = httpx.Timeout(30.0, connect=5.0)
    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        timeout=timeout_config,
        max_retries=2
    )
    with open(input_csv, newline='', encoding="utf-8") as fin, \
         open(output_csv, "w", newline='', encoding="utf-8") as fout:
        reader = csv.DictReader(fin)
        fieldnames = reader.fieldnames + ["llm_label", "llm_raw", "model", "run_at", "prompt_version"]
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()
        for row in reader:
            title = row.get("title", "")
            abstract = row.get("abstract", "")
            if not abstract:
                row["llm_label"], row["llm_raw"] = "EXCLUDE", "EXCLUDE (no abstract)"
            else:
                label, raw = classify_row(client, title, abstract)
                row["llm_label"], row["llm_raw"] = label, raw
            row["model"] = MODEL
            row["run_at"] = datetime.now(timezone.utc).isoformat()
            row["prompt_version"] = "SimpleX-v1"
            writer.writerow(row)
            time.sleep(1 / rate_limit_per_sec)

os.environ["OPENAI_API_KEY"] = "YOUR_SECRET"
run("/content/drive/MyDrive/Colab Notebooks/articles_title_abs.csv", "4_screened_records.csv",rate_limit_per_sec=1)

In [ ]:
with open('screened_records.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

['list', 'Title:', 'title', 'Abstract:\xa0', 'abstract', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'llm_label', 'llm_raw', 'model', 'run_at', 'prompt_version']
['1', 'Title:', 'ServLessSense: Serverless Smell Detection Tool', 'Abstract:\xa0', 'Serverless computing has gained widespread adoption due to its scalability, cost-efficiency, and abstraction of infrastructure management. However, the shift toward event-driven, function-based architectures introduces new code quality challenges and development practices that differ from traditional paradigms. While recent research has identified serverless-specific bad practices commonly referred to as “smells,” there remains a lack of automated tools to support their detection and remediation. This paper presents ServLessSense, a tool designed to detect code smells automatically in serverless applications written in JavaScript and TypeScript. Built using a custom ESLint plugin, ServLessSense identifies five serverless-specifi

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Install deps


In [ ]:
!pip -q install pypdf openai pydantic pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 7.2 MB/s eta 0:00:00


## Read PDF -> text

In [ ]:
from pypdf import PdfReader
import os
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI

PDF_PATH = "/content/drive/MyDrive/artigos/13_RefDiff 2.0: A Multi-Language Refactoring Detection Tool.pdf"

assert os.path.exists(PDF_PATH), f"File not found: {PDF_PATH}"

def pdf_to_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)  # [web:31]
    return "\n".join([(p.extract_text() or "") for p in reader.pages]).strip()  # [web:31]

article_text = pdf_to_text(PDF_PATH)
len(article_text), article_text[:1000]


(90007,
 'IEEE TRANSACTIONS ON SOFTWARE ENGINEERING, VOL. XX, NO. X, AUGUST XXXX 1\nRefDiff 2.0: A Multi-language Refactoring\nDetection Tool\nDanilo Silva, Jo˜ao Paulo da Silva, Gustavo Santos, Ricardo Terra, and Marco Tulio Valente,Member, IEEE\nAbstract—Identifying refactoring operations in source code changes is valuable to understand software evolution. Therefore, several\ntools have been proposed to automatically detect refactorings applied in a system by comparing source code between revisions. The\navailability of such infrastructure has enabled researchers to study refactoring practice in large scale, leading to important advances on\nrefactoring knowledge. However, although a plethora of programming languages are used in practice, the vast majority of existing\nstudies are restricted to the Java language due to limitations of the underlying tools. This fact poses an important threat to external\nvalidity. Thus, to overcome such limitation, in this paper we propose RefDiff 2.0

## Structured extraction


In [ ]:
client = OpenAI()

class Extraction(BaseModel):
    study_title: str
    authors: str
    year_of_publication: str
    source: str

    tool_name: str
    tool_description: str
    technologies_used: str

    code_smells_detected: str
    detection_techniques: str

    validation_type: str
    test_base_dataset_used: str
    performance_metrics: str

    main_results: str
    limitations_identified: str

    application_relevance: str
    additional_observations: str

FIELDS_META = [
    ("Identification", "Study Title", "study_title"),
    ("Identification", "Authors", "authors"),
    ("Identification", "Year of Publication", "year_of_publication"),
    ("Identification", "Source", "source"),
    ("Characteristics", "Tool Name", "tool_name"),
    ("Characteristics", "Tool Description", "tool_description"),
    ("Characteristics", "Technologies Used", "technologies_used"),
    ("Technical Details", "Code Smells Detected", "code_smells_detected"),
    ("Technical Details", "Detection Techniques", "detection_techniques"),
    ("Validation", "Validation Type", "validation_type"),
    ("Validation", "Test Base / Dataset Used", "test_base_dataset_used"),
    ("Validation", "Performance Metrics", "performance_metrics"),
    ("Results", "Main Results", "main_results"),
    ("Results", "Limitations Identified", "limitations_identified"),
    ("Context", "Application / Relevance", "application_relevance"),
    ("Evaluation", "Additional Observations", "additional_observations"),
]

def extract_from_text(text: str, source: str) -> Extraction:
    system_instructions = (
        "Role: You are a Software Engineering Specialist Researcher, focusing on Code Maintenance and Technical Debt.\n"
        "Task: Perform technically accurate data extraction from the provided article text, focusing on code smell "
        "detection tools and methodologies.\n"
        "Rules:\n"
        "- Fill every field.\n"
        "- If information is not explicitly present or clearly implied, write exactly: NOT FOUND\n"
        "- Keep technical terms in their original language (usually English).\n"
    )

    resp = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": f"Article source: {source}\n\nARTICLE TEXT:\n{text}"},
        ],
        text_format=Extraction,
    )
    ex: Extraction = resp.output_parsed
    if (ex.source or "").strip() in ("", "NOT FOUND"):
        ex.source = source
    return ex

def to_long(article_id: str, ex: Extraction) -> pd.DataFrame:
    exd = ex.model_dump()
    return pd.DataFrame([{
        "Article ID": article_id,
        "Category": cat,
        "Information to be Extracted": label,
        "Data Extracted from the Article": exd.get(key, "NOT FOUND"),
    } for (cat, label, key) in FIELDS_META])

ex = extract_from_text(article_text, PDF_PATH)

article_id = os.path.splitext(os.path.basename(PDF_PATH))[0]
df_wide = pd.DataFrame([{**ex.model_dump(), "article_id": article_id, "pdf_path": PDF_PATH}])
df_long = to_long(article_id, ex)

out_path = "/content/drive/MyDrive/artigos/extractions_single.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:  # multi-sheet Excel output [web:43]
    df_wide.to_excel(writer, index=False, sheet_name="wide")
    df_long.to_excel(writer, index=False, sheet_name="long")

out_path


'/content/drive/MyDrive/artigos/extractions_single.xlsx'

## Extract + save


In [ ]:
import os, glob
import pandas as pd
from pydantic import BaseModel
from pypdf import PdfReader
from openai import OpenAI

client = OpenAI()

ART_DIR = "/content/drive/MyDrive/artigos"
pdf_paths = sorted(glob.glob(f"{ART_DIR}/**/*.pdf", recursive=True))

assert len(pdf_paths) > 0, f"No PDFs found in: {ART_DIR}"

def pdf_to_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    return "\n".join([(p.extract_text() or "") for p in reader.pages]).strip()

# Schema WITHOUT authors/year/source
class Extraction(BaseModel):
    study_title: str

    tool_name: str
    tool_description: str
    technologies_used: str

    code_smells_detected: str
    detection_techniques: str

    validation_type: str
    test_base_dataset_used: str
    performance_metrics: str

    main_results: str
    limitations_identified: str

    application_relevance: str
    additional_observations: str

FIELDS_META = [
    ("Identification", "Study Title", "study_title"),

    ("Characteristics", "Tool Name", "tool_name"),
    ("Characteristics", "Tool Description", "tool_description"),
    ("Characteristics", "Technologies Used", "technologies_used"),

    ("Technical Details", "Code Smells Detected", "code_smells_detected"),
    ("Technical Details", "Detection Techniques", "detection_techniques"),

    ("Validation", "Validation Type", "validation_type"),
    ("Validation", "Test Base / Dataset Used", "test_base_dataset_used"),
    ("Validation", "Performance Metrics", "performance_metrics"),

    ("Results", "Main Results", "main_results"),
    ("Results", "Limitations Identified", "limitations_identified"),

    ("Context", "Application / Relevance", "application_relevance"),
    ("Evaluation", "Additional Observations", "additional_observations"),
]

SYSTEM_INSTRUCTIONS = (
    "Role: You are a Software Engineering Specialist Researcher, focusing on Code Maintenance and Technical Debt.\n"
    "Task: Perform technically accurate data extraction from the provided article text, focusing on code smell "
    "detection tools and methodologies.\n"
    "Rules:\n"
    "- Fill every field.\n"
    "- If information is not explicitly present or clearly implied, write exactly: NOT FOUND\n"
    "- Keep technical terms in their original language (usually English).\n"
)

def extract_from_text(text: str, source: str) -> Extraction:
    resp = client.responses.parse(
        model="gpt-4o-mini",  # change to "gpt-4o-2024-08-06" if you want max accuracy
        input=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": f"Article source: {source}\n\nARTICLE TEXT:\n{text}"},
        ],
        text_format=Extraction,
    )
    return resp.output_parsed

def to_long(article_id: str, ex: Extraction) -> pd.DataFrame:
    exd = ex.model_dump()
    return pd.DataFrame([{
        "Article ID": article_id,
        "PDF Path": exd_path,
        "Category": cat,
        "Information to be Extracted": label,
        "Data Extracted from the Article": exd.get(key, "NOT FOUND"),
    } for (cat, label, key) in FIELDS_META])

wide_rows = []
long_dfs = []

for idx, exd_path in enumerate(pdf_paths, start=1):
    article_id = os.path.splitext(os.path.basename(exd_path))[0]

    text = pdf_to_text(exd_path)
    if not text:
        # Still create a row, but everything will become NOT FOUND
        text = ""

    ex = extract_from_text(text, source=exd_path)

    wide = ex.model_dump()
    wide.update({"article_id": article_id, "pdf_path": exd_path})
    wide_rows.append(wide)

    # reuse exd_path inside to_long
    long_dfs.append(to_long(article_id, ex))

df_wide = pd.DataFrame(wide_rows)
df_long = pd.concat(long_dfs, ignore_index=True)

out_path = f"{ART_DIR}/extractions_all.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_wide.to_excel(writer, index=False, sheet_name="wide")
    df_long.to_excel(writer, index=False, sheet_name="long")

out_path


'/content/drive/MyDrive/artigos/extractions_all.xlsx'